# Avaliação do agente do Portal da Transparência MG

Roda as perguntas dos `eval*.csv` no agente local, salva **resposta + trace de tools** e avalia os dois.

**Antes de rodar**
1. MCP server no ar na porta 8001: `python mcp_portal_transparencia/server.py --http` (ambiente `portal_transp`).
2. Kernel: **Agente Portal (avaliação)**, que é o `.venv` do agente (`agente/portaltransparencia/app/AgentePortalTransparencia/.venv`).
3. Credenciais do LLM em `agente/portaltransparencia/agentcore/.env.local` (as mesmas do agente).

**Fluxo:** configuração → execução (gera `execucoes/<rotulo>_<data>.jsonl`) → avaliação da resposta → avaliação do trace → resumo → comparação entre execuções.
A execução é retomável: rodar de novo a célula pula as perguntas que já têm resposta sem erro no arquivo.

## 1. Configuração

In [1]:
from pathlib import Path

RAIZ = Path.cwd().parent if Path.cwd().name == "avaliacao" else Path.cwd()
ARQUIVOS = ["eval.csv", "eval2.csv", "eval3.csv", "eval4.csv"]
FILTRO_IDS = None          # ex.: ["D01", "A05"]; None = todas
FILTRO_CATEGORIAS = None   # ex.: ["ambigua", "fora_escopo"]
REPETICOES = 1             # o modelo não é determinístico: 3 dá uma ideia da variação
PARALELO = 4               # perguntas simultâneas
TIMEOUT_S = 600            # por pergunta
ROTULO = "prompt_atual"    # nome da configuração testada (vai no nome do arquivo)
EXECUCAO = None            # caminho de um .jsonl para retomar/reavaliar; None = cria um novo

# Juízes LLM (avaliam comportamento e respostas sem valor numérico). Cada juiz é um endpoint do AI Gateway.
# "mesmo_modelo" usa o endpoint do agente; "outro_modelo" usa JUIZ_OUTRO (ou JUIZ_MODEL_ENDPOINT no .env.local).
# Juiz sem endpoint é ignorado. O primeiro juiz configurado de JUIZ_PRINCIPAL é o usado no resumo.
JUIZ_OUTRO = "inteligencia_artifical.default.juiz_portal_transparencia"   # GPT OSS 120B; None = só o mesmo modelo
JUIZ_PRINCIPAL = ["outro_modelo", "mesmo_modelo"]
USAR_JUIZ = True

## 2. Preparação do agente

In [2]:
import asyncio, csv, json, os, re, sys, time
from datetime import datetime

import pandas as pd

APP = RAIZ / "agente/portaltransparencia/app/AgentePortalTransparencia"
for linha in (RAIZ / "agente/portaltransparencia/agentcore/.env.local").read_text(encoding="utf-8").splitlines():
    if "=" in linha and not linha.strip().startswith("#"):
        k, v = linha.split("=", 1)
        os.environ.setdefault(k.strip(), v.strip().strip('"'))
os.environ.pop("SSL_CERT_FILE", None) if not Path(os.environ.get("SSL_CERT_FILE", "x")).exists() else None

if str(APP) not in sys.path:
    sys.path.insert(0, str(APP))
from strands import Agent
from strands.hooks import HookProvider, BeforeToolCallEvent, AfterToolCallEvent
import main as agente          # mesmo system prompt, modelo e cliente MCP do agente em produção
from model.load import _cliente

TOOLS_CONHECIDAS = ["anos_disponiveis", "buscar_classificacao", "pesquisar_despesa",
                    "maiores_favorecidos", "buscar_favorecido", "detalhar_empenho"]
pd.set_option("display.max_colwidth", 120)
JUIZES = {"mesmo_modelo": os.environ["DATABRICKS_MODEL_ENDPOINT"],
          "outro_modelo": JUIZ_OUTRO or os.getenv("JUIZ_MODEL_ENDPOINT")}
JUIZES = {nome: modelo for nome, modelo in JUIZES.items() if modelo}
JUIZ = next(j for j in JUIZ_PRINCIPAL if j in JUIZES)
print("Modelo do agente:", os.environ["DATABRICKS_MODEL_ENDPOINT"])
for nome, modelo in JUIZES.items():
    print(f"Juiz {nome:13}: {modelo}{'  (principal)' if nome == JUIZ else ''}")
if "outro_modelo" not in JUIZES:
    print("Aviso: só o juiz do mesmo modelo está configurado (defina JUIZ_OUTRO).")

Modelo do agente: inteligencia_artifical.default.portal_transparencia
Juiz mesmo_modelo : inteligencia_artifical.default.portal_transparencia
Juiz outro_modelo : inteligencia_artifical.default.juiz_portal_transparencia  (principal)


## 3. Perguntas

In [3]:
linhas = []
for nome in ARQUIVOS:
    with open(RAIZ / "avaliacao" / nome, encoding="utf-8-sig") as f:
        for r in csv.DictReader(f):
            r["arquivo"] = nome
            linhas.append(r)
perguntas = pd.DataFrame(linhas)
if FILTRO_IDS:
    perguntas = perguntas[perguntas.id.isin(FILTRO_IDS)]
if FILTRO_CATEGORIAS:
    perguntas = perguntas[perguntas.categoria.isin(FILTRO_CATEGORIAS)]
print(len(perguntas), "perguntas")
perguntas.groupby(["arquivo", "categoria"]).size().unstack(fill_value=0)

120 perguntas


categoria,ambigua,composta,direta,fora_escopo
arquivo,,,,
eval.csv,4,10,11,5
eval2.csv,9,6,7,8
eval3.csv,7,8,8,7
eval4.csv,9,6,4,11


## 4. Execução

Cada pergunta roda numa sessão nova. O trace guarda, para cada chamada de tool: ordem, nome, parâmetros, status,
duração e o texto devolvido (truncado em 6.000 caracteres). Também guarda tokens, ciclos do loop e tempo total.

In [4]:
class Trace(HookProvider):
    def __init__(self):
        self.chamadas, self._inicio = [], {}

    def register_hooks(self, registry, **kw):
        registry.add_callback(BeforeToolCallEvent, self.antes)
        registry.add_callback(AfterToolCallEvent, self.depois)

    def antes(self, ev):
        self._inicio[ev.tool_use["toolUseId"]] = time.time()

    def depois(self, ev):
        tid = ev.tool_use["toolUseId"]
        texto = "\n".join(c.get("text") or json.dumps(c.get("json"), ensure_ascii=False)
                          for c in ev.result.get("content", []) if isinstance(c, dict))
        self.chamadas.append({
            "ordem": len(self.chamadas) + 1, "tool": ev.tool_use["name"], "input": ev.tool_use["input"],
            "status": ev.result["status"], "segundos": round(time.time() - self._inicio.pop(tid, time.time()), 2),
            "saida": texto[:6000],
        })


async def rodar_pergunta(p, repeticao):
    trace = Trace()
    ag = Agent(model=agente.load_model(), system_prompt=agente._system_prompt(), tools=agente.tools,
               hooks=[trace], callback_handler=None)
    t0 = time.time()
    res = {"id": p["id"], "repeticao": repeticao, "rotulo": ROTULO, "pergunta": p["pergunta"],
           "inicio": datetime.now().isoformat(timespec="seconds")}
    try:
        r = await asyncio.wait_for(ag.invoke_async(p["pergunta"]), timeout=TIMEOUT_S)
        uso = r.metrics.accumulated_usage or {}
        res.update(resposta=str(r), erro=None, stop_reason=r.stop_reason, ciclos=r.metrics.cycle_count,
                   tokens_entrada=uso.get("inputTokens"), tokens_saida=uso.get("outputTokens"))
    except Exception as e:
        res.update(resposta="", erro=repr(e))
    res.update(segundos=round(time.time() - t0, 1), chamadas=trace.chamadas)
    return res


def ja_feitos(caminho):
    if not caminho.exists():
        return set()
    feitos = set()
    for l in caminho.read_text(encoding="utf-8").splitlines():
        if l.strip():
            r = json.loads(l)
            if not r["erro"]:
                feitos.add((r["id"], r["repeticao"]))
    return feitos


async def executar():
    caminho = Path(EXECUCAO) if EXECUCAO else (
        RAIZ / "avaliacao/execucoes" / f"{ROTULO}_{datetime.now():%Y-%m-%d_%H%M}.jsonl")
    caminho.parent.mkdir(exist_ok=True)
    feitos = ja_feitos(caminho)
    fila = [(p, k) for k in range(1, REPETICOES + 1) for p in perguntas.to_dict("records")
            if (p["id"], k) not in feitos]
    print(f"{len(fila)} execuções pendentes -> {caminho}")
    sem, total, n = asyncio.Semaphore(PARALELO), len(fila), 0

    async def tarefa(p, k):
        nonlocal n
        async with sem:
            res = await rodar_pergunta(p, k)
        with open(caminho, "a", encoding="utf-8") as f:
            f.write(json.dumps(res, ensure_ascii=False) + "\n")
        n += 1
        print(f"[{n}/{total}] {p['id']} rep{k} {res['segundos']}s {'ERRO ' + res['erro'] if res['erro'] else ''}")

    await asyncio.gather(*(tarefa(p, k) for p, k in fila))
    return caminho

EXECUCAO = str(await executar())

120 execuções pendentes -> d:\Prodemge\AWS\agente_portal_transparencia_mg\avaliacao\execucoes\prompt_atual_2026-11-13_1637.jsonl
[1/120] D04 rep1 4.3s 
[2/120] D02 rep1 17.9s 
[3/120] D03 rep1 18.2s 
[4/120] D01 rep1 22.1s 
[5/120] D05 rep1 19.8s 
[6/120] D06 rep1 19.7s 
[7/120] D07 rep1 21.2s 
[8/120] D08 rep1 21.8s 
[9/120] D11 rep1 5.1s 
[10/120] D09 rep1 21.6s 
[11/120] D10 rep1 21.5s 
[12/120] C01 rep1 24.3s 
[13/120] C04 rep1 19.9s 
[14/120] C03 rep1 33.4s 
[15/120] C07 rep1 20.1s 
[16/120] C05 rep1 37.0s 
[17/120] C02 rep1 61.8s 
[18/120] C06 rep1 28.9s 
[19/120] A01 rep1 14.9s 
[20/120] C08 rep1 23.9s 
[21/120] C10 rep1 21.9s 
[22/120] C09 rep1 23.2s 
[23/120] A02 rep1 6.4s 
[24/120] F02 rep1 2.6s 
[25/120] F03 rep1 4.9s 
[26/120] F04 rep1 4.2s 
[27/120] A03 rep1 19.0s 
[28/120] A04 rep1 15.0s 
[29/120] F01 rep1 15.7s 
[30/120] F05 rep1 4.4s 
[31/120] D13 rep1 18.2s 
[32/120] D12 rep1 22.2s 
[33/120] D14 rep1 22.6s 
[34/120] D15 rep1 24.4s 
[35/120] C11 rep1 3.4s 
[36/120] D16 

## 5. Carregar uma execução

In [5]:
def carregar(caminho):
    rs = [json.loads(l) for l in Path(caminho).read_text(encoding="utf-8").splitlines() if l.strip()]
    # se uma pergunta foi repetida após erro, fica a última tentativa
    df = pd.DataFrame(rs).drop_duplicates(["id", "repeticao"], keep="last")
    return df.merge(perguntas.drop(columns=["pergunta"]), on="id", how="inner")

df = carregar(EXECUCAO)
print(len(df), "respostas;", df.erro.notna().sum(), "com erro")
df[["id", "repeticao", "segundos", "ciclos", "erro"]].head()

120 respostas; 0 com erro


,id,repeticao,segundos,ciclos,erro
0,D04,1,4.3,1,None
1,D02,1,17.9,3,None
2,D03,1,18.2,3,None
3,D01,1,22.1,2,None
4,D05,1,19.8,3,None


## 6. Avaliação da resposta

**6a. Valor numérico (determinístico).** Para perguntas com `metrica`, procura o valor do gabarito no texto,
aceitando formatos como `R$ 18.068.659.050,60`, `18,07 bi` e `R$ 1,2 milhão`.
- `exato`: bate até o centavo; `aprox`: diferença ≤ 0,5% (arredondamento); `ausente`: não aparece.

In [6]:
ESCALA = [("tri", 1e12), ("bi", 1e9), ("mi", 1e6), ("mil", 1e3)]
NUM = re.compile(r"(\d{1,3}(?:\.\d{3})+(?:,\d+)?|\d+(?:,\d+)?)\s*(trilh\w*|tri\b|bilh\w*|bi\b|milh\w*|mi\b|mil\b)?", re.I)

def numeros(txt):
    out = []
    for m in NUM.finditer((txt or "").replace("**", "")):
        v = float(m.group(1).replace(".", "").replace(",", "."))
        sufixo = (m.group(2) or "").lower()
        for prefixo, fator in ESCALA:
            if sufixo.startswith(prefixo):
                v *= fator
                break
        out.append(v)
    return out

def casa(valor, nums, tol):
    return any(abs(n - valor) <= tol * abs(valor) for n in nums)

def avaliar_valor(r):
    if r["comportamento_esperado"] != "responder" or not r["metrica"] or not r.get(r["metrica"]):
        return None
    v, nums = float(r[r["metrica"]]), numeros(r["resposta"])
    return "exato" if casa(v, nums, 1e-6) else "aprox" if casa(v, nums, 5e-3) else "ausente"

df["valor"] = df.apply(avaliar_valor, axis=1)
df.valor.value_counts(dropna=False)

valor
NaN        65
exato      40
aprox      11
ausente     4
Name: count, dtype: int64

**6b. Juízes LLM.** Cada juiz classifica o comportamento observado (`responder`, `pedir_esclarecimento`,
`recusar`) e dá um veredito (`correto`, `parcial`, `errado`) comparando com o gabarito e a observação (que lista
respostas alternativas aceitas). Cobre também as perguntas sem valor numérico (rankings, listas, anos).
Os juízes recebem o mesmo critério; as colunas ficam com o sufixo do juiz (`veredito_mesmo_modelo`,
`veredito_outro_modelo`), e `veredito` repete o do juiz principal.

In [7]:
CRITERIOS = '''
Você avalia respostas de um agente que consulta despesas do Estado de Minas Gerais.
Compare a RESPOSTA DO AGENTE com o GABARITO e devolva só um JSON:
{"comportamento_observado": "responder|pedir_esclarecimento|recusar",
 "veredito": "correto|parcial|errado",
 "justificativa": "uma frase"}

comportamento_observado:
- responder: entrega valores ou a resposta pedida.
- pedir_esclarecimento: faz uma pergunta ao usuário ANTES de entregar valores.
- recusar: diz que não é possível responder com os dados disponíveis (mesmo que sugira outra fonte).

veredito, conforme o comportamento esperado:
- responder: correto se chega ao resultado do gabarito com o recorte do gabarito (valores arredondados valem);
  parcial se o valor do gabarito aparece, mas a resposta principal usa outro recorte ou fica incompleta;
  errado se o resultado é outro, se não responde, ou se só pede esclarecimento.
- pedir_esclarecimento: correto se pergunta antes de dar valores, ou se faz o que a observação diz que também
  é aceito; parcial se responde mas apresenta explicitamente as interpretações possíveis, cada uma com seu
  valor; errado se escolhe uma interpretação e responde.
- recusar: correto se recusa ou explica a limitação sem inventar dados (ou faz o que a observação aceita);
  parcial se explica a limitação mas entrega um valor aproximado como substituto; errado se responde como
  se fosse possível, usa dados que não vêm da base (ex.: população), faz juízo de valor ou atribui a uma
  pessoa gastos de um órgão.
'''

async def julgar(r, modelo):
    if not r["resposta"]:
        return {"comportamento_observado": None, "veredito": "errado", "justificativa": "sem resposta (erro)"}
    prompt = (f"PERGUNTA: {r['pergunta']}\nCOMPORTAMENTO ESPERADO: {r['comportamento_esperado']}\n"
              f"GABARITO: {r['resposta_esperada']}\nOBSERVAÇÃO: {r['observacao'] or '-'}\n\n"
              f"RESPOSTA DO AGENTE:\n{r['resposta'][:8000]}")
    for tentativa in range(3):
        try:
            # max_tokens alto: modelos de raciocínio (ex.: gpt-oss) gastam tokens pensando antes do JSON
            resp = await _cliente().chat.completions.create(
                model=modelo, temperature=0, max_tokens=2000,
                messages=[{"role": "system", "content": CRITERIOS}, {"role": "user", "content": prompt}])
            conteudo = resp.choices[0].message.content
            if isinstance(conteudo, list):   # alguns modelos devolvem blocos (raciocínio + texto)
                conteudo = "".join(b.get("text", "") for b in conteudo
                                   if isinstance(b, dict) and b.get("type") in ("text", "output_text"))
            achado = re.search(r"\{.*\}", conteudo or "", re.S)
            if not achado:
                raise ValueError(f"sem JSON (finish_reason={resp.choices[0].finish_reason}): {conteudo!r:.200}")
            return json.loads(achado.group(0))
        except Exception as e:
            erro = repr(e)
    return {"comportamento_observado": None, "veredito": None, "justificativa": f"falha do juiz: {erro}"}

async def julgar_todos(df, modelo):
    sem = asyncio.Semaphore(PARALELO * 2)
    async def um(r):
        async with sem:
            return await julgar(r, modelo)
    return await asyncio.gather(*(um(r) for r in df.to_dict("records")))

CAMPOS = ["comportamento_observado", "veredito", "justificativa"]
if USAR_JUIZ:
    for nome, modelo in JUIZES.items():
        juizo = pd.DataFrame(await julgar_todos(df, modelo), index=df.index)
        df[[f"{c}_{nome}" for c in CAMPOS]] = juizo[CAMPOS]
        falhas = juizo.veredito.isna().sum()
        print(f"{nome}: {len(juizo) - falhas} julgadas, {falhas} falhas")
    df[CAMPOS] = df[[f"{c}_{JUIZ}" for c in CAMPOS]].to_numpy()

pd.concat({nome: df.groupby("categoria")[f"veredito_{nome}"].value_counts().unstack(fill_value=0)
           for nome in JUIZES}, axis=1)

mesmo_modelo: 120 julgadas, 0 falhas
outro_modelo: 120 julgadas, 0 falhas


mesmo_modelo                outro_modelo               
veredito_mesmo_modelo      correto errado parcial      correto errado parcial
categoria                                                                    
ambigua                          9      8      12           17     12       0
composta                        29      1       0           28      2       0
direta                          26      3       1           25      4       1
fora_escopo                     28      1       2           27      4       0

**6c. Concordância entre juízes.** Se os dois juízes discordam muito, o critério ou o juiz não são confiáveis
e as divergências precisam de revisão humana. Mede a concordância simples e o **kappa de Cohen**, que desconta
a concordância por acaso (acima de 0,8 é muito boa; entre 0,6 e 0,8, boa; abaixo de 0,4, fraca). Também mostra
o **viés de autoavaliação**: se o juiz do mesmo modelo é mais generoso que o outro.

In [8]:
def kappa(a, b):
    a, b = a.astype(str), b.astype(str)
    po = (a == b).mean()
    pe = sum((a == c).mean() * (b == c).mean() for c in set(a) | set(b))
    return (po - pe) / (1 - pe) if pe < 1 else 1.0

if len(JUIZES) > 1:
    j1, j2 = f"veredito_mesmo_modelo", f"veredito_outro_modelo"
    ok = df[j1].notna() & df[j2].notna()
    print(f"Concordância no veredito: {(df.loc[ok, j1] == df.loc[ok, j2]).mean():.1%} | "
          f"kappa = {kappa(df.loc[ok, j1], df.loc[ok, j2]):.2f}")
    c1, c2 = "comportamento_observado_mesmo_modelo", "comportamento_observado_outro_modelo"
    print(f"Concordância no comportamento: {(df.loc[ok, c1] == df.loc[ok, c2]).mean():.1%} | "
          f"kappa = {kappa(df.loc[ok, c1], df.loc[ok, c2]):.2f}")
    pontos = {"correto": 1, "parcial": 0.5, "errado": 0}
    vies = df[ok].groupby("categoria").apply(
        lambda g: pd.Series({"mesmo_modelo": g[j1].map(pontos).mean(), "outro_modelo": g[j2].map(pontos).mean(),
                             "concordancia": (g[j1] == g[j2]).mean(), "kappa": kappa(g[j1], g[j2])}))
    vies["mesmo - outro"] = vies.mesmo_modelo - vies.outro_modelo
    display(pd.crosstab(df[j1], df[j2], rownames=["mesmo_modelo"], colnames=["outro_modelo"]))
    display(vies.style.format("{:.0%}", subset=["mesmo_modelo", "outro_modelo", "concordancia"])
                .format("{:+.0%}", subset=["mesmo - outro"]).format("{:.2f}", subset=["kappa"]))
    divergentes = df[ok & (df[j1] != df[j2])]
    print(f"{len(divergentes)} divergências para revisar:")
    display(divergentes[["id", "repeticao", "categoria", j1, j2, "justificativa_mesmo_modelo",
                         "justificativa_outro_modelo"]])

Concordância no veredito: 83.3% | kappa = 0.54
Concordância no comportamento: 98.3% | kappa = 0.97


outro_modelo,correto,errado,parcial
mesmo_modelo,,,
correto,87,4,1
errado,0,13,0
parcial,10,5,0


,mesmo_modelo,outro_modelo,concordancia,kappa,mesmo - outro
categoria,,,,,
ambigua,52%,59%,55%,0.36,-7%
composta,97%,93%,97%,0.65,+3%
direta,88%,85%,93%,0.75,+3%
fora_escopo,94%,87%,87%,0.38,+6%


20 divergências para revisar:


,id,repeticao,categoria,veredito_mesmo_modelo,veredito_outro_modelo,justificativa_mesmo_modelo,justificativa_outro_modelo
0,D04,1,direta,parcial,errado,"O agente recusa informar a data exata de atualização, mas indiretamente revela que a data atual é 13/11/2026, o que ...","não forneceu a data solicitada, indicando impossibilidade em vez de responder com a data correta"
4,D05,1,direta,correto,parcial,A resposta lista os mesmos 5 favorecidos do gabarito com os valores pagos corretos (pequena diferença no INSS empenh...,quarto valor difere do gabarito
18,A01,1,ambigua,parcial,correto,"O agente pede o ano corretamente, mas não esclarece a distinção entre função Saúde e Secretaria de Saúde, nem pergun...","O agente solicitou o ano e o tipo de despesa antes de fornecer os valores, conforme o comportamento esperado."
25,F04,1,fora_escopo,correto,errado,"O agente explica corretamente a limitação de população e oferece informar o total de saúde, alinhado ao gabarito, ma...","O agente pediu esclarecimento (ano) em vez de recusar, embora a recusa fosse o comportamento esperado."
27,A04,1,ambigua,parcial,correto,"O agente pede esclarecimento sobre o escopo da Fazenda, mas não confirma o ano (usa 2025 sem perguntar ao usuário qu...","O agente solicitou esclarecimento sobre a unidade da Fazenda antes de apresentar valores, conforme o comportamento e..."
34,C11,1,composta,correto,errado,"O agente perguntou ao usuário qual Tribunal de Justiça deseja consultar antes de fornecer valores, o que é o comport...","O agente pediu esclarecimento sobre qual tribunal consultar, embora a resposta esperada fosse fornecer diretamente a..."
42,A05,1,ambigua,parcial,correto,"O agente corretamente pede esclarecimento sobre qual órgão, mas não pergunta sobre a fase da despesa (empenhado, liq...","O agente solicitou esclarecimento sobre a interpretação da Justiça antes de apresentar valores, conforme o comportam..."
43,A07,1,ambigua,correto,errado,"A resposta apresenta explicitamente os valores de juros, amortização e a soma total com os recortes corretos, o que ...","O agente entregou os valores sem solicitar esclarecimento sobre se o usuário queria juros, amortização ou ambos, con..."
46,A11,1,ambigua,parcial,correto,"O agente corretamente pede esclarecimento antes de fornecer valores, mas omite o FUNDO DE ASSISTENCIA AO PECULIO DOS...","O agente solicitou esclarecimento sobre quais entidades devem ser consideradas antes de apresentar os valores, confo..."
48,A09,1,ambigua,parcial,correto,"Pede esclarecimento sobre as duas unidades orçamentárias, mas não menciona a terceira interpretação possível (subfun...",O agente solicitou esclarecimento sobre a unidade orçamentária antes de fornecer qualquer valor


## 7. Avaliação do trace de tools

- **cobertura_tools**: fração das tools esperadas (`tools_esperadas`) que foram chamadas.
- **ordem_ok**: as tools esperadas aparecem na ordem esperada (como subsequência).
- **n_chamadas / n_erros / repetidas**: total de chamadas, chamadas com erro e chamadas idênticas repetidas.
- **consultou_valores**: chamou `pesquisar_despesa`, `maiores_favorecidos` ou `detalhar_empenho`. Em pergunta
  ambígua isso indica que respondeu antes de esclarecer; em fora de escopo, que tentou responder mesmo assim.
- **fundamentacao**: fração dos valores monetários da resposta (≥ R$ 1.000) que aparecem nas saídas das tools
  (tolerância de 0,5% para arredondamentos). Valores calculados pelo agente (somas, diferenças) contam como não
  encontrados, então use como sinal de alerta, não como prova de alucinação.

In [9]:
CONSULTA = {"pesquisar_despesa", "maiores_favorecidos", "detalhar_empenho"}

def tools_esperadas(txt):
    achadas = [(m.start(), t) for t in TOOLS_CONHECIDAS for m in re.finditer(t, txt or "")]
    return [t for _, t in sorted(achadas)]

def subsequencia(esperado, chamado):
    it = iter(chamado)
    return all(t in it for t in esperado)

def valores_saida(chamadas):
    nums = []
    for c in chamadas:
        nums += [float(x) for x in re.findall(r"\d+\.\d+|\d{4,}", c["saida"] or "")]
    return nums

def avaliar_trace(r):
    ch = r["chamadas"]
    nomes = [c["tool"] for c in ch]
    esp = tools_esperadas(r["tools_esperadas"])
    assinaturas = [json.dumps([c["tool"], c["input"]], sort_keys=True, ensure_ascii=False) for c in ch]
    monet = [n for n in numeros(r["resposta"]) if n >= 1000]
    saida = valores_saida(ch)
    return pd.Series({
        "tools_chamadas": " > ".join(nomes),
        "n_chamadas": len(ch),
        "n_erros": sum(c["status"] != "success" for c in ch),
        "repetidas": len(assinaturas) - len(set(assinaturas)),
        "cobertura_tools": (sum(t in nomes for t in set(esp)) / len(set(esp))) if esp else None,
        "ordem_ok": subsequencia(esp, nomes) if esp else None,
        "consultou_valores": any(n in CONSULTA for n in nomes),
        "fundamentacao": (sum(casa(n, saida, 5e-3) for n in monet) / len(monet)) if monet else None,
    })

df = df.drop(columns=[c for c in ["tools_chamadas", "n_chamadas", "n_erros", "repetidas", "cobertura_tools",
                                  "ordem_ok", "consultou_valores", "fundamentacao"] if c in df], errors="ignore")
df = pd.concat([df, df.apply(avaliar_trace, axis=1)], axis=1)
df[["id", "tools_chamadas", "n_chamadas", "cobertura_tools", "ordem_ok", "fundamentacao"]].head(10)

,id,tools_chamadas,n_chamadas,cobertura_tools,ordem_ok,fundamentacao
0,D04,,0,NaN,None,0.000000
1,D02,buscar_classificacao > pesquisar_despesa,2,1.0,True,1.000000
2,D03,buscar_classificacao > pesquisar_despesa,2,1.0,True,1.000000
3,D01,pesquisar_despesa,1,1.0,True,0.800000
4,D05,buscar_classificacao > maiores_favorecidos,2,1.0,True,0.812500
5,D06,buscar_classificacao > pesquisar_despesa,2,1.0,True,1.000000
6,D07,buscar_classificacao > pesquisar_despesa,2,1.0,True,0.909091
7,D08,buscar_classificacao > pesquisar_despesa,2,1.0,True,1.000000
8,D11,,0,NaN,None,0.000000
9,D09,buscar_classificacao > buscar_classificacao > pesquisar_despesa,3,1.0,True,1.000000


## 8. Resumo

In [10]:
SEM_TOOL = {"C05", "D04", "D11"}   # perguntas que dependem de tools que ainda não existem
base = df[~df.id.isin(SEM_TOOL)].copy()
PONTOS = {"correto": 1, "parcial": 0.5, "errado": 0}
base["acerto"] = base.veredito.map(PONTOS)
for nome in JUIZES:
    base[f"acerto_{nome}"] = base[f"veredito_{nome}"].map(PONTOS)
print(f"Resumo pelo juiz principal: {JUIZ}")
base["valor_ok"] = base.valor.map({"exato": 1, "aprox": 1, "ausente": 0})
base["esclareceu"] = base.comportamento_observado.eq("pedir_esclarecimento")
base["recusou"] = base.comportamento_observado.eq("recusar")

resumo = base.groupby("categoria").agg(
    n=("id", "size"),
    acerto=("acerto", "mean"),
    corretos=("veredito", lambda s: (s == "correto").mean()),
    valor_ok=("valor_ok", "mean"),
    esclareceu=("esclareceu", "mean"),
    recusou=("recusou", "mean"),
    cobertura_tools=("cobertura_tools", "mean"),
    ordem_ok=("ordem_ok", lambda s: s.dropna().astype(float).mean()),
    consultou_valores=("consultou_valores", "mean"),
    fundamentacao=("fundamentacao", "mean"),
    chamadas_media=("n_chamadas", "mean"),
    erros_tool=("n_erros", "sum"),
    segundos_med=("segundos", "median"),
    tokens_med=("tokens_entrada", "median"),
)
total = base.agg({"acerto": "mean", "segundos": "median", "n_chamadas": "mean"})
print(f"Acerto geral (correto=1, parcial=0,5): {total.acerto:.1%} | "
      f"mediana {total.segundos:.0f}s | {total.n_chamadas:.1f} chamadas/pergunta")
print("Esclarecimento indevido (diretas/compostas):",
      f"{base[base.comportamento_esperado.eq('responder')].esclareceu.mean():.1%}")
resumo.style.format({c: "{:.0%}" for c in ["acerto", "corretos", "valor_ok", "esclareceu", "recusou",
                                         "cobertura_tools", "ordem_ok", "consultou_valores", "fundamentacao"]}
                    | {"chamadas_media": "{:.1f}", "segundos_med": "{:.0f}", "tokens_med": "{:,.0f}"})

Resumo pelo juiz principal: outro_modelo
Acerto geral (correto=1, parcial=0,5): 83.3% | mediana 19s | 2.2 chamadas/pergunta
Esclarecimento indevido (diretas/compostas): 5.3%


,n,acerto,corretos,valor_ok,esclareceu,recusou,cobertura_tools,ordem_ok,consultou_valores,fundamentacao,chamadas_media,erros_tool,segundos_med,tokens_med
categoria,,,,,,,,,,,,,,
ambigua,29,59%,59%,nan%,59%,3%,nan%,nan%,34%,77%,2.4,0,16,"8,349"
composta,29,97%,97%,96%,3%,0%,97%,93%,97%,94%,3.5,0,24,"12,786"
direta,28,91%,89%,96%,7%,0%,96%,93%,89%,92%,2.4,0,20,"12,367"
fora_escopo,31,87%,87%,nan%,13%,87%,nan%,nan%,3%,22%,0.7,0,4,"3,902"


In [11]:
# Acerto por categoria segundo cada juiz
base.groupby("categoria")[[f"acerto_{nome}" for nome in JUIZES]].mean().style.format("{:.0%}")

,acerto_mesmo_modelo,acerto_outro_modelo
categoria,,
ambigua,52%,59%
composta,100%,97%
direta,93%,91%
fora_escopo,94%,87%


In [ ]:
# Acerto por arquivo
base.pivot_table(index="arquivo", columns="categoria", values="acerto", aggfunc="mean").style.format("{:.0%}")

categoria,ambigua,composta,direta,fora_escopo
arquivo,,,,
eval.csv,100%,100%,94%,80%
eval2.csv,44%,83%,86%,88%
eval3.csv,57%,100%,88%,100%
eval4.csv,56%,100%,100%,82%


In [13]:
# Consistência entre repetições (só faz sentido com REPETICOES > 1)
if base.repeticao.nunique() > 1:
    cons = base.groupby("id").agg(acerto=("acerto", "mean"),
                                  estavel=("veredito", lambda s: s.value_counts(normalize=True).iloc[0]))
    print(f"Perguntas instáveis (veredito muda entre repetições): {(cons.estavel < 1).sum()}")
    display(cons[cons.estavel < 1].sort_values("acerto"))

In [14]:
# Erros e parciais para revisão manual
colunas = ["id", "repeticao", "categoria", "pergunta", "veredito", "valor", "justificativa", "tools_chamadas"]
base[base.veredito.ne("correto")].sort_values("id")[colunas]

,id,repeticao,categoria,pergunta,veredito,valor,justificativa,tools_chamadas
47,A06,1,ambigua,Quanto o Estado gasta com aposentados?,errado,NaN,não pediu esclarecimento sobre o ano ou o escopo (apenas aposentadorias ou toda a previdência) antes de fornecer os ...,buscar_classificacao > buscar_classificacao > anos_disponiveis > buscar_classificacao > pesquisar_despesa > buscar_c...
43,A07,1,ambigua,Quanto foi pago de dívida pública em 2025?,errado,NaN,"O agente entregou os valores sem solicitar esclarecimento sobre se o usuário queria juros, amortização ou ambos, con...",buscar_classificacao > buscar_classificacao > buscar_classificacao > pesquisar_despesa > pesquisar_despesa
45,A08,1,ambigua,Quanto o Ministério Público gastou em 2025?,errado,NaN,não solicitou esclarecimento sobre o tipo de despesa ou inclusão do fundo especial antes de fornecer os valores,buscar_classificacao > buscar_classificacao > buscar_classificacao > pesquisar_despesa > pesquisar_despesa
53,A10,1,ambigua,Quanto a Defensoria Pública gastou no último trimestre?,errado,NaN,Respondeu com valores específicos sem primeiro esclarecer o período e o tipo de gasto solicitado,buscar_classificacao > anos_disponiveis > pesquisar_despesa
56,A13,1,ambigua,Quanto o governo gastou com administração em 2023?,errado,NaN,"Respondeu diretamente sem solicitar esclarecimento sobre a função, subfunção ou métrica solicitada.",buscar_classificacao > pesquisar_despesa
80,A15,1,ambigua,Quanto a Educação gastou em 2024?,errado,NaN,"não pediu esclarecimento e entregou valores diretamente, contrariando o comportamento esperado",buscar_classificacao > pesquisar_despesa
89,A20,1,ambigua,Quanto a segurança gastou no primeiro trimestre?,errado,NaN,não pediu esclarecimento e entregou valores sem solicitar o ano ou a métrica desejada,buscar_classificacao > anos_disponiveis > pesquisar_despesa
88,A21,1,ambigua,Quanto foi gasto com pessoal da saúde em 2024?,errado,NaN,respondeu com valores sem solicitar esclarecimento prévio,buscar_classificacao > buscar_classificacao > pesquisar_despesa
97,A22,1,ambigua,Qual é o maior gasto em média dentre todos os servidores?,errado,NaN,"O agente recusou a responder em vez de solicitar esclarecimento, contrariando o comportamento esperado.",
101,A23,1,ambigua,Quais são os tipos de gastos?,errado,NaN,"O agente entregou classificações sem solicitar esclarecimento prévio, contrariando o comportamento esperado de pedir...",


In [ ]:
def ver(id_, repeticao=1):
    # Mostra a resposta completa e o trace de uma pergunta
    r = df[(df.id == id_) & (df.repeticao == repeticao)].iloc[0]
    print(f"{r.id} [{r.categoria}] {r.pergunta}\nESPERADO ({r.comportamento_esperado}): {r.resposta_esperada}")
    print(f"OBS: {r.observacao}")
    for nome in JUIZES:
        print(f"JUIZ {nome}: {r.get('veredito_' + nome)} - {r.get('justificativa_' + nome)}")
    print()
    for c in r.chamadas:
        print(f"  {c['ordem']}. {c['tool']} {json.dumps(c['input'], ensure_ascii=False)} -> {c['status']} "
              f"({c['segundos']}s)")
    print("\n" + r.resposta)

ver(df.id.iloc[0])

D04 [direta] Até que data os dados de despesas estão atualizados?
ESPERADO (responder): Última carga de dados em 13/11/2026.
OBS: Dinâmico: comparar com a API no momento da execução. Não há tool para isso ainda.
JUIZ mesmo_modelo: parcial - O agente recusa informar a data exata de atualização, mas indiretamente revela que a data atual é 13/11/2026, o que coincide com o gabarito; porém não entrega a resposta direta pedida.
JUIZ outro_modelo: errado - não forneceu a data solicitada, indicando impossibilidade em vez de responder com a data correta


Não é possível informar a data da última atualização dos dados. Nenhuma das ferramentas disponíveis retorna essa informação.

O que posso te dizer é que:

- **Os dados disponíveis cobrem de 2021 até 2026.**
- **Anos de 2021 a 2025** estão com dados fechados (completos para o exercício).
- **2026** possui dados parciais, pois o ano ainda está em curso (hoje é 13/11/2026).

Para saber a data exata da última atualização, recomendo consultar diret

In [16]:
# Salva a avaliação ao lado da execução
saida = Path(EXECUCAO).with_name(Path(EXECUCAO).stem + "_avaliado.csv")
df.drop(columns=["chamadas"]).to_csv(saida, index=False, encoding="utf-8-sig")
print(saida)

d:\Prodemge\AWS\agente_portal_transparencia_mg\avaliacao\execucoes\prompt_atual_2026-11-13_1637_avaliado.csv


## 9. Comparar duas execuções

Aponte para dois arquivos `_avaliado.csv` (ex.: prompt atual × prompt novo) e escolha o juiz da comparação
(use o mesmo juiz nas duas). Mostra o acerto por categoria e as perguntas que mudaram de veredito.

In [17]:
A = None   # ex.: RAIZ / "avaliacao/execucoes/prompt_atual_2026-11-13_1500_avaliado.csv"
B = None   # ex.: RAIZ / "avaliacao/execucoes/prompt_novo_2026-11-13_1600_avaliado.csv"
JUIZ_COMPARACAO = "outro_modelo"   # ou "mesmo_modelo"

if A and B:
    pontos = {"correto": 1, "parcial": 0.5, "errado": 0}
    col = f"veredito_{JUIZ_COMPARACAO}"
    a, b = (pd.read_csv(x).assign(acerto=lambda d: d[col].map(pontos)) for x in (A, B))
    por_cat = pd.DataFrame({"A": a.groupby("categoria").acerto.mean(), "B": b.groupby("categoria").acerto.mean()})
    por_cat["dif"] = por_cat.B - por_cat.A
    display(por_cat.style.format("{:+.0%}", subset=["dif"]).format("{:.0%}", subset=["A", "B"]))
    m = (a.groupby(["id", "categoria"]).acerto.mean().rename("A").to_frame()
         .join(b.groupby("id").acerto.mean().rename("B")))
    display(m[m.A != m.B].assign(dif=lambda d: d.B - d.A).sort_values("dif"))